# STAIR-v3: ClipFuse — kNN-Discriminability Confidence + Topology-Preserving Binary Graph

**Bugs fixed from STAIR-v2 (DyFuse):**
- **L1 — Confidence collapse:** Raw dim-wise std → visual (0.942) >> text (0.044) due to scale mismatch → c_v=95.6%, text suppressed. **v3 fix:** Use `disc_m = 1 - mean_knn_sim_m` (lower mean kNN cosine sim → items more spread → more discriminative → higher confidence).
- **L2 — Device mismatch crash:** `torch.ones(..., device=cfg.device)` created CUDA tensor while edge_index was CPU → crash in `to_undirected`. **v3 fix:** all graph ops on CPU, `register_buffer` handles device transfer.
- **L3 — Python loop crash:** `_topk_binarize` used Python for-loop over CUDA tensor elements → slow/crash. **v3 fix:** fully vectorized cumsum-based rank computation.
- **L4 — Continuous weights:** Cosine edge weights skew BSC diffusion. **v3 fix:** TopK then BINARIZE → binary 0/1 adjacency.
- **L5 — Modality collapse:** Clip confidence to [δ, 1-δ] (δ=0.3) → both modalities always contribute ≥ 30%.

**Key property:** 0 extra learnable parameters — only changes `mAdj` construction in `prepare()`.

| | |
|---|---|
| Dataset | Amazon2014Baby + Amazon2014Sports |
| Epochs | 500 |
| Confidence | kNN-Discriminability: `1 - mean_kNN_cosine_sim` |
| δ (floor) | 0.3 (min weight per modality) |
| τ (temperature) | 1.0 |

In [ ]:
import os, shutil

os.chdir('/kaggle/working')

# Clone / refresh repo
repo = 'STAIR-Enhanced'
if os.path.exists(repo):
    shutil.rmtree(repo)
os.system('git clone https://github.com/ThanhChuong12/STAIR-Enhanced.git')

# Install dependencies
os.system('pip install nvidia-ml-py -q')
os.system('pip install torchdata==0.6.1 --no-deps -q')
os.system('pip install freerec==0.9.7 -q')
os.system('pip install torch_geometric -q')
os.system('pip install prettytable -q')

import torch, platform, freerec
print('Python  :', platform.python_version())
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
print('freerec :', freerec.__version__)
print('Environment ready')


In [ ]:
# Cell 2: Copy dataset files from /kaggle/input to /kaggle/data
import os, shutil

DATA_ROOT = '/kaggle/data'
os.makedirs(DATA_ROOT, exist_ok=True)

def copy_dataset(keywords, full_name):
    dest = os.path.join(DATA_ROOT, full_name)
    os.makedirs(dest, exist_ok=True)
    copied = []
    for root, _, files in os.walk('/kaggle/input'):
        if any(kw.lower() in root.lower() for kw in keywords):
            for f in files:
                if f.endswith(('.npy', '.pkl', '.txt', '.inter', '.item')):
                    shutil.copy(os.path.join(root, f), os.path.join(dest, f))
                    copied.append(f)
    print(f'[{full_name}] {len(copied)} files copied')

copy_dataset(['baby', 'amazon2014baby'],     'Amazon2014Baby_550_MMRec')
copy_dataset(['sports', 'amazon2014sports'], 'Amazon2014Sports_550_MMRec')
print('Data ready at', DATA_ROOT)


In [ ]:
# Cell 3: ClipFuse Confidence Diagnostics — kNN-Discriminability vs v2 (DyFuse)
# Shows how v3 fixes the confidence collapse problem in v2
import os, sys, pickle, warnings, math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

sys.path.insert(0, '/kaggle/working/STAIR-Enhanced')
warnings.filterwarnings('ignore', category=RuntimeWarning)

DATA_ROOT = '/kaggle/data'
DATASETS  = ['Amazon2014Baby_550_MMRec', 'Amazon2014Sports_550_MMRec']
MFILES    = ['textual_modality.pkl', 'visual_modality.pkl']

def load_feat(ds, mf):
    with open(os.path.join(DATA_ROOT, ds, mf), 'rb') as f:
        feat = pickle.load(f)
    if not isinstance(feat, torch.Tensor):
        feat = torch.tensor(feat, dtype=torch.float32)
    return feat.float()

def build_knn_edge_weights(feat, k):
    """Build kNN graph and return top-k cosine similarities."""
    feat_n = F.normalize(feat, p=2, dim=-1)
    sim = feat_n @ feat_n.t()
    sim.fill_diagonal_(-10.)
    topk_vals, _ = sim.topk(k=k, dim=-1)
    return topk_vals.clamp(min=0.).reshape(-1)  # (N*k,)

def compute_v2_confidence(ft, fv):
    """STAIR-v2: dim-wise std on RAW features. Collapsed due to scale bias."""
    eps = 1e-7
    std_v = fv.std(dim=0).mean().item()
    std_t = ft.std(dim=0).mean().item()
    c_v_raw = std_v / (std_v + std_t + eps)
    return c_v_raw, 1.0 - c_v_raw, std_v, std_t

def compute_v3_confidence(ew_t, ew_v, delta=0.3, tau=1.0):
    """STAIR-v3: kNN-discriminability. Lower mean kNN sim -> more spread -> higher confidence."""
    eps = 1e-7
    mean_sim_t = ew_t.mean().item()
    mean_sim_v = ew_v.mean().item()
    disc_t = max(0., 1. - mean_sim_t)
    disc_v = max(0., 1. - mean_sim_v)
    exp_v   = math.exp(disc_v / (tau + eps))
    exp_t   = math.exp(disc_t / (tau + eps))
    c_v_raw = exp_v / (exp_v + exp_t + eps)
    c_v = max(delta, min(1.0 - delta, c_v_raw))
    c_t = 1.0 - c_v
    return c_v_raw, c_v, c_t, mean_sim_v, mean_sim_t, disc_v, disc_t

K_TEXT, K_VIS = 5, 1   # default num-neighbors

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle('STAIR-v3 (ClipFuse) — Confidence: kNN-Discriminability vs. v2 (DyFuse)',
             fontsize=13, fontweight='bold')

for ri, ds in enumerate(DATASETS):
    ds_name = 'Baby' if 'Baby' in ds else 'Sports'
    try:
        ft = load_feat(ds, MFILES[0])
        fv = load_feat(ds, MFILES[1])
    except FileNotFoundError:
        print(f'[WARN] Files not found for {ds_name}. Run Cell 2 first.')
        continue

    nt = ft.norm(p=2, dim=-1).numpy()
    nv = fv.norm(p=2, dim=-1).numpy()

    # v2 confidence (scale-biased)
    cv2_raw, ct2, std_v, std_t = compute_v2_confidence(ft, fv)

    # v3: build kNN graphs first, then compute discriminability
    print(f'[{ds_name}] Building kNN graphs (k_t={K_TEXT}, k_v={K_VIS})...')
    ew_t = build_knn_edge_weights(ft, K_TEXT)
    ew_v = build_knn_edge_weights(fv, K_VIS)
    cv3_raw, cv3, ct3, mean_sv, mean_st, disc_v, disc_t = compute_v3_confidence(ew_t, ew_v)

    print(f'  Raw norms  : text={nt.mean():.4f}, visual={nv.mean():.4f}')
    print(f'  v2 (DyFuse): raw_std_v={std_v:.4f}, raw_std_t={std_t:.4f} => c_v={cv2_raw:.4f} [COLLAPSED]')
    print(f'  v3 (ClipFuse): mean_knn_sim_v={mean_sv:.4f}, mean_knn_sim_t={mean_st:.4f}')
    print(f'    disc_v={disc_v:.4f}, disc_t={disc_t:.4f} => raw_c_v={cv3_raw:.4f} -> clipped c_v={cv3:.4f}')
    print(f'  => v3: Visual {cv3*100:.1f}%, Text {ct3*100:.1f}%')

    ax = axes[ri]

    # --- Plot 1: L2-norm distribution ---
    ax[0].hist(nv, bins=60, color='#E8734A', alpha=0.85, label='Visual L2-norm', density=True)
    ax[0].axvline(nt.mean(), color='#4A90D9', linewidth=2.5, linestyle='--',
                  label=f'Text L2-norm={nt.mean():.2f} (fixed)')
    ax[0].set_title(f'{ds_name} — L2-Norm Distribution', fontweight='bold')
    ax[0].set_xlabel('L2 Norm'); ax[0].set_ylabel('Density')
    ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)

    # --- Plot 2: Confidence comparison (v2 vs v3) ---
    x, width = np.arange(2), 0.3
    b2 = ax[1].bar(x - width/2, [ct2, cv2_raw], width, color=['#4A90D9','#E8734A'],
                    alpha=0.45, label='v2 (raw std, collapsed)', edgecolor='gray', hatch='//')
    b3 = ax[1].bar(x + width/2, [ct3, cv3],     width, color=['#4A90D9','#E8734A'],
                    alpha=0.9,  label='v3 (kNN disc. + clip)', edgecolor='white')
    ax[1].axhline(0.3, color='red', linewidth=1.5, linestyle=':', label='delta=0.3 floor')
    ax[1].set_title(f'{ds_name} — Confidence: v2 vs v3', fontweight='bold')
    ax[1].set_xticks(x); ax[1].set_xticklabels(['Text (c_t)', 'Visual (c_v)'])
    ax[1].set_ylabel('Confidence Weight'); ax[1].set_ylim(0, 1.2)
    for bar, val in zip(b2, [ct2, cv2_raw]):
        ax[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                   f'{val:.3f}', ha='center', va='bottom', fontsize=9, color='gray')
    for bar, val in zip(b3, [ct3, cv3]):
        ax[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                   f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax[1].legend(fontsize=7); ax[1].grid(alpha=0.3, axis='y')

    # --- Plot 3: kNN cosine similarity distribution (v3 confidence basis) ---
    ax[2].hist(ew_t.numpy(), bins=60, alpha=0.75, color='#4A90D9',
               label=f'Text kNN (k={K_TEXT}, mean={mean_st:.3f})', density=True)
    ax[2].hist(ew_v.numpy(), bins=60, alpha=0.75, color='#E8734A',
               label=f'Visual kNN (k={K_VIS}, mean={mean_sv:.3f})', density=True)
    ax[2].axvline(mean_st, color='#2471A3', linewidth=2, linestyle='--')
    ax[2].axvline(mean_sv, color='#A04000', linewidth=2, linestyle='--')
    ax[2].set_title(f'{ds_name} — kNN Cosine Similarity Dist. (v3 confidence basis)', fontweight='bold')
    ax[2].set_xlabel('Cosine Similarity'); ax[2].set_ylabel('Density')
    ax[2].legend(fontsize=7); ax[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/stair_v3_confidence_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nv3 confidence uses kNN-discriminability: 1 - mean_kNN_sim.')
print('Lower mean sim = items more spread = more discriminative = higher confidence.')


In [ ]:
# Cell 4: Train STAIR-v3 (ClipFuse) on Amazon2014Baby
import os, subprocess, time, shutil

os.chdir('/kaggle/working/STAIR-Enhanced')

cmd_baby = [
    'python', 'main_v3.py',
    '--root', '/kaggle/data',
    '--dataset', 'Amazon2014Baby_550_MMRec',
    '--epochs', '500',
    '--batch-size', '1024',
    '--embedding-dim', '64',
    '--num-layers', '3',
    '--num-neighbors', '5-1',
    '--conf-delta', '0.3',
    '--conf-temp', '1.0',
    '--optimizer', 'adamwsevo',
    '--lr', '1e-3',
    '--weight-decay', '0.1',
    '--seed', '1',
]

log_path_baby = '/kaggle/working/log_stair_v3_baby.txt'

print('Training STAIR-ClipFuse on Baby...')
t0 = time.time()
result = subprocess.run(
    cmd_baby,
    stdout=open(log_path_baby, 'w'),
    stderr=subprocess.STDOUT,
    text=True
)
elapsed = (time.time() - t0) / 60
print(f'Done in {elapsed:.1f} min | rc={result.returncode}')

# Show last 20 lines of log for quick sanity check
with open(log_path_baby) as f:
    lines = f.readlines()
if result.returncode != 0:
    print('\n[ERROR] Training failed! Last 30 lines of log:')
    print(''.join(lines[-30:]))
else:
    print('\nLast 5 log lines:')
    print(''.join(lines[-5:]))

os.makedirs('/kaggle/working/STAIR-Enhanced/logs', exist_ok=True)
shutil.copy(log_path_baby, '/kaggle/working/STAIR-Enhanced/logs/log_stair_v3_baby.txt')


In [ ]:
# Cell 5: Train STAIR-v3 (ClipFuse) on Amazon2014Sports
import os, subprocess, time, shutil

os.chdir('/kaggle/working/STAIR-Enhanced')

cmd_sports = [
    'python', 'main_v3.py',
    '--root', '/kaggle/data',
    '--dataset', 'Amazon2014Sports_550_MMRec',
    '--epochs', '500',
    '--batch-size', '1024',
    '--embedding-dim', '64',
    '--num-layers', '3',
    '--num-neighbors', '5-1',
    '--conf-delta', '0.3',
    '--conf-temp', '1.0',
    '--optimizer', 'adamwsevo',
    '--lr', '1e-3',
    '--weight-decay', '0.1',
    '--seed', '1',
]

log_path_sports = '/kaggle/working/log_stair_v3_sports.txt'

print('Training STAIR-ClipFuse on Sports...')
t0 = time.time()
result = subprocess.run(
    cmd_sports,
    stdout=open(log_path_sports, 'w'),
    stderr=subprocess.STDOUT,
    text=True
)
elapsed = (time.time() - t0) / 60
print(f'Done in {elapsed:.1f} min | rc={result.returncode}')

with open(log_path_sports) as f:
    lines = f.readlines()
if result.returncode != 0:
    print('\n[ERROR] Training failed! Last 30 lines of log:')
    print(''.join(lines[-30:]))
else:
    print('\nLast 5 log lines:')
    print(''.join(lines[-5:]))

os.makedirs('/kaggle/working/STAIR-Enhanced/logs', exist_ok=True)
shutil.copy(log_path_sports, '/kaggle/working/STAIR-Enhanced/logs/log_stair_v3_sports.txt')


In [ ]:
# Cell 6: Parse training logs and show best metrics
import re, os

def parse_log(log_path):
    losses, recalls10, recalls20, ndcgs10, ndcgs20 = [], [], [], [], []
    best = {}

    try:
        with open(log_path) as f:
            content = f.read()
    except FileNotFoundError:
        print(f'[WARN] Log not found: {log_path}')
        return losses, recalls10, recalls20, ndcgs10, ndcgs20, best

    # Training loss per epoch
    for m in re.finditer(r'(?i)epoch.*?loss.*?([0-9]+\.[0-9]+)', content):
        try: losses.append(float(m.group(1)))
        except ValueError: pass

    # Validation metrics (multi-line)
    for m in re.finditer(
        r'Recall@10.*?([0-9]+\.[0-9]+).*?Recall@20.*?([0-9]+\.[0-9]+)'
        r'.*?NDCG@10.*?([0-9]+\.[0-9]+).*?NDCG@20.*?([0-9]+\.[0-9]+)',
        content, re.DOTALL
    ):
        try:
            recalls10.append(float(m.group(1))); recalls20.append(float(m.group(2)))
            ndcgs10.append(float(m.group(3)));   ndcgs20.append(float(m.group(4)))
        except ValueError: pass

    # Best metrics
    best_match = re.search(
        r'(?i)best.*?Recall@10.*?([0-9]+\.[0-9]+).*?Recall@20.*?([0-9]+\.[0-9]+)'
        r'.*?NDCG@10.*?([0-9]+\.[0-9]+).*?NDCG@20.*?([0-9]+\.[0-9]+)',
        content, re.DOTALL
    )
    if best_match:
        best = {
            'Recall@10': float(best_match.group(1)), 'Recall@20': float(best_match.group(2)),
            'NDCG@10':   float(best_match.group(3)), 'NDCG@20':   float(best_match.group(4)),
        }
    elif ndcgs20:
        idx = ndcgs20.index(max(ndcgs20))
        best = {'Recall@10': recalls10[idx], 'Recall@20': recalls20[idx],
                'NDCG@10': ndcgs10[idx],     'NDCG@20': ndcgs20[idx]}

    return losses, recalls10, recalls20, ndcgs10, ndcgs20, best

log_baby_v3   = '/kaggle/working/log_stair_v3_baby.txt'
log_sports_v3 = '/kaggle/working/log_stair_v3_sports.txt'

print('Parsing Baby V3...')
loss_baby_v3, r10_baby_v3, r20_baby_v3, n10_baby_v3, n20_baby_v3, best_baby_v3 = parse_log(log_baby_v3)
print(f'  loss={len(loss_baby_v3)} epochs, val={len(r10_baby_v3)} epochs')
print(f'  best={best_baby_v3}')

print('Parsing Sports V3...')
loss_sports_v3, r10_sports_v3, r20_sports_v3, n10_sports_v3, n20_sports_v3, best_sports_v3 = parse_log(log_sports_v3)
print(f'  loss={len(loss_sports_v3)} epochs, val={len(r10_sports_v3)} epochs')
print(f'  best={best_sports_v3}')


In [ ]:
# Cell 7: Learning Curves
import numpy as np
import matplotlib.pyplot as plt

def smooth(values, w=10):
    if len(values) < w: return values
    return np.convolve(values, np.ones(w)/w, mode='valid').tolist()

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle('STAIR-v3 (ClipFuse) — Learning Curves', fontsize=15, fontweight='bold')

for ci, (ds_name, losses, r10, r20, n10, n20) in enumerate([
    ('Baby',   loss_baby_v3,   r10_baby_v3,   r20_baby_v3,   n10_baby_v3,   n20_baby_v3),
    ('Sports', loss_sports_v3, r10_sports_v3, r20_sports_v3, n10_sports_v3, n20_sports_v3),
]):
    ax = axes[0][ci]
    if losses:
        s = smooth(losses, w=10)
        ax.plot(range(len(losses)), losses, color='#E8C55A', alpha=0.25, linewidth=0.8)
        offset = (10 - 1) // 2
        ax.plot(range(offset, offset + len(s)), s, color='#E8734A', linewidth=2, label='Smoothed')
    ax.set_title(f'{ds_name} -- BPR Loss', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = axes[1][ci]
    if r10: ax.plot(r10, color='#2ECC71', linewidth=1.8, label='Recall@10')
    if r20: ax.plot(r20, color='#27AE60', linewidth=1.8, label='Recall@20')
    ax.set_title(f'{ds_name} -- Recall', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Recall'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = axes[2][ci]
    if n10: ax.plot(n10, color='#3498DB', linewidth=1.8, label='NDCG@10')
    if n20: ax.plot(n20, color='#2980B9', linewidth=1.8, label='NDCG@20')
    ax.set_title(f'{ds_name} -- NDCG', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('NDCG'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/stair_v3_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Learning curves saved.')


In [ ]:
# Cell 8: Full Performance Comparison: Baseline vs v1 vs v2 vs v3
from prettytable import PrettyTable

BASELINE = {
    'Baby':   {'Recall@10': 0.068560, 'Recall@20': 0.103420, 'NDCG@10': 0.036335, 'NDCG@20': 0.045143},
    'Sports': {'Recall@10': 0.074310, 'Recall@20': 0.111900, 'NDCG@10': 0.040200, 'NDCG@20': 0.050050},
}
V1_GCL = {
    'Baby':   {'Recall@10': 0.069459, 'Recall@20': 0.104700, 'NDCG@10': 0.036535, 'NDCG@20': 0.045531},
    'Sports': {'Recall@10': 0.074541, 'Recall@20': 0.112394, 'NDCG@10': 0.040429, 'NDCG@20': 0.050400},
}
V2_DYFUSE = {
    'Baby':   {'Recall@10': 0.057800, 'Recall@20': 0.089800, 'NDCG@10': 0.030900, 'NDCG@20': 0.038800},
    'Sports': {'Recall@10': 0.066900, 'Recall@20': 0.100200, 'NDCG@10': 0.036600, 'NDCG@20': 0.044900},
}
V3_CLIPFUSE = {
    'Baby':   best_baby_v3   if best_baby_v3   else {'Recall@10': 0, 'Recall@20': 0, 'NDCG@10': 0, 'NDCG@20': 0},
    'Sports': best_sports_v3 if best_sports_v3 else {'Recall@10': 0, 'Recall@20': 0, 'NDCG@10': 0, 'NDCG@20': 0},
}

METRICS  = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']
DATA_MAP = [BASELINE, V1_GCL, V2_DYFUSE, V3_CLIPFUSE]

print('=' * 115)
print('COMPARISON: Baseline  vs  v1 (GCL)  vs  v2 (DyFuse)  vs  v3 (ClipFuse)')
print('=' * 115)

for ds in ['Baby', 'Sports']:
    t = PrettyTable()
    t.field_names = ['Metric', 'Baseline', 'v1 (GCL)', 'v2 (DyFuse)', 'v3 (ClipFuse)',
                     'v3 vs Base', 'v3 vs v1', 'v3 vs v2']
    for metric in METRICS:
        vals = [data[ds].get(metric, 0) for data in DATA_MAP]
        v3 = vals[3]
        t.add_row([
            metric,
            f'{vals[0]:.6f}', f'{vals[1]:.6f}', f'{vals[2]:.6f}', f'{vals[3]:.6f}',
            f'{(v3-vals[0])/(vals[0]+1e-9)*100:+.2f}%',
            f'{(v3-vals[1])/(vals[1]+1e-9)*100:+.2f}%',
            f'{(v3-vals[2])/(vals[2]+1e-9)*100:+.2f}%',
        ])
    print(f'\nDataset: {ds}'); print(t)

print('=' * 115)


In [ ]:
# Cell 9: Bar Chart Comparison
import numpy as np
import matplotlib.pyplot as plt

METRICS  = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']
COLORS   = ['#95A5A6', '#3498DB', '#E74C3C', '#2ECC71']
LABELS   = ['Baseline', 'v1 (GCL)', 'v2 (DyFuse)', 'v3 (ClipFuse)']
DATA_MAP = [BASELINE, V1_GCL, V2_DYFUSE, V3_CLIPFUSE]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Model Comparison — Baseline vs v1 (GCL) vs v2 (DyFuse) vs v3 (ClipFuse)',
             fontsize=13, fontweight='bold')

for ri, ds in enumerate(['Baby', 'Sports']):
    for ci, metric in enumerate(METRICS):
        ax = axes[ri][ci]
        vals = [data[ds].get(metric, 0) for data in DATA_MAP]
        bars = ax.bar(LABELS, vals, color=COLORS, alpha=0.85, edgecolor='white', width=0.6)
        ax.set_title(f'{ds} — {metric}', fontweight='bold', fontsize=10)
        ax.set_ylim(0, max(vals) * 1.2)
        ax.tick_params(axis='x', rotation=25, labelsize=7)
        ax.grid(alpha=0.3, axis='y')
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                    f'{val:.4f}', ha='center', va='bottom', fontsize=8,
                    fontweight='bold' if val == max(vals) else 'normal')

plt.tight_layout()
plt.savefig('/kaggle/working/stair_v3_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Cell 10: Improvement Heatmap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

METRICS  = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']
comparisons = [
    ('v1 vs Base', V1_GCL, BASELINE), ('v2 vs Base', V2_DYFUSE, BASELINE),
    ('v3 vs Base', V3_CLIPFUSE, BASELINE), ('v3 vs v1', V3_CLIPFUSE, V1_GCL),
    ('v3 vs v2', V3_CLIPFUSE, V2_DYFUSE),
]
rows = [f'{ds} — {m}' for ds in ['Baby', 'Sports'] for m in METRICS]
cols = [c[0] for c in comparisons]
data = np.zeros((len(rows), len(cols)))

for ci, (_, model, ref) in enumerate(comparisons):
    ri = 0
    for ds in ['Baby', 'Sports']:
        for metric in METRICS:
            m_val = model.get(ds, {}).get(metric, 0)
            r_val = ref.get(ds, {}).get(metric, 1e-9)
            data[ri, ci] = (m_val - r_val) / (r_val + 1e-9) * 100
            ri += 1

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(data, cmap='RdYlGn', aspect='auto',
               norm=mcolors.TwoSlopeNorm(vmin=-20, vcenter=0, vmax=10))
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, fontsize=10)
ax.set_yticks(range(len(rows))); ax.set_yticklabels(rows, fontsize=9)
plt.colorbar(im, ax=ax, label='Relative Improvement (%)')
for i in range(len(rows)):
    for j in range(len(cols)):
        val = data[i, j]
        ax.text(j, i, f'{val:+.1f}%', ha='center', va='center', fontsize=9,
                fontweight='bold', color='white' if abs(val) > 8 else 'black')
ax.set_title('Relative Improvement (%) Heatmap', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/stair_v3_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Cell 11: Final Summary + Output File List
import os, shutil

print('=' * 70)
print('STAIR-v3 ClipFuse — Final Results')
print('=' * 70)

for ds, best in [('Baby', best_baby_v3), ('Sports', best_sports_v3)]:
    base = BASELINE[ds]
    print(f'\n[{ds}]')
    if best:
        for metric in ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']:
            v3 = best.get(metric, 0)
            bl = base[metric]
            d  = (v3 - bl) / (bl + 1e-9) * 100
            print(f'  {metric:<12}: {v3:.6f}  (Baseline={bl:.6f}, delta={d:+.2f}%)')
    else:
        print('  [No results — check training log]')

print('=' * 70)

los_dir = '/kaggle/working/STAIR-Enhanced/logs'
os.makedirs(los_dir, exist_ok=True)

output_files = [
    '/kaggle/working/log_stair_v3_baby.txt',
    '/kaggle/working/log_stair_v3_sports.txt',
    '/kaggle/working/stair_v3_confidence_diagnostics.png',
    '/kaggle/working/stair_v3_learning_curves.png',
    '/kaggle/working/stair_v3_comparison.png',
    '/kaggle/working/stair_v3_heatmap.png',
]

print('\nOutput Files:')
for fp in output_files:
    if os.path.exists(fp):
        size_kb = os.path.getsize(fp) / 1024
        print(f'  [OK]   {os.path.basename(fp):<50} {size_kb:.1f} KB')
        dst = os.path.join(los_dir, os.path.basename(fp))
        if fp != dst: shutil.copy(fp, dst)
    else:
        print(f'  [MISS] {os.path.basename(fp)}')
